# CNN-GRU Blood Glucose Prediction from PPG Signals

This notebook implements the complete training pipeline for non-invasive glucose monitoring.

**Architecture**: CNN-GRU Hybrid  
**Target Performance**: MAE < 3.0 mg/dL, R² > 0.95  
**Dataset**: VitalDB or MUST

Based on: "Non-Invasive Glucose Level Monitoring from PPG using a Hybrid CNN-GRU Deep Learning Network" (2024)

## 1. Setup and Installation

In [ ]:
# Install dependencies (if needed on Kaggle)
!pip install neurokit2 vitaldb -q

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.append('../src')

from models import build_cnn_gru
from preprocessing import PPGPreprocessor
from training import train_model
from evaluation import compute_all_metrics, print_metrics, clarke_error_grid_analysis

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load VitalDB Dataset

**VitalDB** is an open dataset with 6,388 surgical patients containing:
- **PPG waveforms**: `SNUADC/PLETH` at 500 Hz
- **Glucose measurements**: Laboratory results (preoperative and perioperative)

We'll use the VitalDB Python API to access the data without manual downloads.

### VitalDB Track Information:
- **SNUADC/PLETH**: Plethysmography waveform (500 Hz)
- **Clinical data**: Includes `preop_gluc` (preoperative glucose in mg/dL)
- **Lab results**: Time-series glucose measurements during surgery

In [ ]:
# Load VitalDB dataset using Python API
import vitaldb
from tqdm import tqdm

print("Loading VitalDB dataset...")
print("This may take several minutes on first run.\n")

# Step 1: Find all cases with PPG (PLETH) data
print("Step 1: Finding cases with PPG signals...")
ppg_cases = vitaldb.find_cases(['SNUADC/PLETH'])
print(f"Found {len(ppg_cases)} cases with PPG data")

# Step 2: Load clinical data for glucose values
print("\nStep 2: Loading clinical data with glucose measurements...")
clinical_df = pd.read_csv('https://api.vitaldb.net/clinical_data.csv')

# Filter for cases with preoperative glucose data
glucose_df = clinical_df[['caseid', 'preop_gluc']].dropna()
print(f"Found {len(glucose_df)} cases with preoperative glucose data")

# Step 3: Find intersection - cases with both PPG and glucose
common_cases = list(set(ppg_cases) & set(glucose_df['caseid'].values))
print(f"\nStep 3: Found {len(common_cases)} cases with both PPG and glucose data")

# Limit to subset for faster training (adjust as needed)
MAX_CASES = 200  # Increase for full dataset
if len(common_cases) > MAX_CASES:
    common_cases = np.random.choice(common_cases, MAX_CASES, replace=False)
    print(f"Using subset of {MAX_CASES} cases for training")

# Step 4: Load PPG signals and glucose values
print(f"\nStep 4: Loading PPG signals from {len(common_cases)} cases...")

ppg_signals = []
glucose_values = []
patient_ids = []
sampling_rate = 500  # VitalDB PLETH sampling rate

for caseid in tqdm(common_cases, desc="Loading cases"):
    try:
        # Load PPG signal at 100Hz (downsample from 500Hz)
        vals = vitaldb.load_case(caseid, ['SNUADC/PLETH'], 1/100)
        
        if vals is None or len(vals) == 0:
            continue
            
        ppg_signal = vals[:, 0]
        
        # Get glucose value for this patient
        glucose = glucose_df[glucose_df['caseid'] == caseid]['preop_gluc'].values[0]
        
        # Split long signal into multiple 5-second segments
        segment_length = 500  # 5 seconds at 100Hz
        n_segments = len(ppg_signal) // segment_length
        
        for i in range(n_segments):
            start_idx = i * segment_length
            end_idx = start_idx + segment_length
            segment = ppg_signal[start_idx:end_idx]
            
            if len(segment) == segment_length:
                ppg_signals.append(segment)
                glucose_values.append(glucose)
                patient_ids.append(caseid)
        
    except Exception as e:
        print(f"Error loading case {caseid}: {e}")
        continue

ppg_signals = np.array(ppg_signals)
glucose_values = np.array(glucose_values)
patient_ids = np.array(patient_ids)

print(f"\n{'='*60}")
print(f"VitalDB Dataset Loaded Successfully")
print(f"{'='*60}")
print(f"Total samples: {len(ppg_signals)}")
print(f"Unique patients: {len(np.unique(patient_ids))}")
print(f"PPG signal shape: {ppg_signals[0].shape} (5 seconds at 100Hz)")
print(f"Glucose range: {glucose_values.min():.1f} - {glucose_values.max():.1f} mg/dL")
print(f"Glucose mean ± std: {glucose_values.mean():.1f} ± {glucose_values.std():.1f} mg/dL")
print(f"{'='*60}")

# Note: Actual sampling rate is now 100Hz after downsampling
sampling_rate = 100

## 3. Preprocess PPG Signals

In [ ]:
# Initialize preprocessor
preprocessor = PPGPreprocessor(
    sampling_rate=sampling_rate,
    segment_length=1.0,  # 1-second segments
    lowcut=0.5,
    highcut=8.0,
    similarity_threshold=0.85
)

print("Preprocessing PPG signals...")

# Preprocess all signals
processed_segments = []
processed_glucose = []
processed_patient_ids = []

for ppg, glucose, patient_id in zip(ppg_signals, glucose_values, patient_ids):
    # Preprocess and segment
    segments = preprocessor.preprocess(ppg, apply_template_matching=True)
    
    # Each segment gets the same glucose value
    for segment in segments:
        processed_segments.append(segment)
        processed_glucose.append(glucose)
        processed_patient_ids.append(patient_id)

X = np.array(processed_segments)  # (n_samples, 100)
y = np.array(processed_glucose)
patient_ids_processed = np.array(processed_patient_ids)

print(f"\nPreprocessed data:")
print(f"  Total segments: {len(X)}")
print(f"  Segment shape: {X[0].shape}")
print(f"  Average segments per original sample: {len(X) / len(ppg_signals):.1f}")

## 4. Train/Test Split by Patient

In [ ]:
from sklearn.model_selection import train_test_split

# Split by patient ID to prevent data leakage
unique_patients = np.unique(patient_ids_processed)
print(f"Total unique patients: {len(unique_patients)}")

train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    random_state=42
)

# Further split train into train/val
train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.2,
    random_state=42
)

# Create masks
train_mask = np.isin(patient_ids_processed, train_patients)
val_mask = np.isin(patient_ids_processed, val_patients)
test_mask = np.isin(patient_ids_processed, test_patients)

# Split data
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"\nData split:")
print(f"  Train: {len(X_train)} samples from {len(train_patients)} patients")
print(f"  Val: {len(X_val)} samples from {len(val_patients)} patients")
print(f"  Test: {len(X_test)} samples from {len(test_patients)} patients")

# Verify no patient overlap
assert len(set(train_patients) & set(val_patients)) == 0
assert len(set(train_patients) & set(test_patients)) == 0
assert len(set(val_patients) & set(test_patients)) == 0
print("\nData split verification: No patient overlap")

## 5. Build Model

In [ ]:
# Build CNN-GRU model
model = build_cnn_gru(
    input_length=100,  # 1 second at 100Hz
    dropout=0.3
)

print(f"Model built successfully")
print(f"Total parameters: {model.count_parameters():,}")
print(f"\nModel architecture:")
print(model)

## 6. Train Model

In [ ]:
# Training configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
epochs = 100
learning_rate = 0.001

# Train model
trainer = train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    device=device,
    save_dir='checkpoints'
)

## 7. Plot Training History

In [ ]:
history = trainer.history

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE plot
axes[1].plot(history['train_mae'], label='Train MAE', linewidth=2)
axes[1].plot(history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE (mg/dL)', fontsize=12)
axes[1].set_title('Training and Validation MAE', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBest validation MAE: {min(history['val_mae']):.2f} mg/dL at epoch {np.argmin(history['val_mae'])+1}")

## 8. Evaluate on Test Set

In [ ]:
# Load best model
trainer.load_checkpoint('checkpoints/best_model.pt')

# Make predictions
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred = model(X_test_tensor).cpu().numpy().squeeze()

# Compute metrics
metrics = compute_all_metrics(y_test, y_pred)
print_metrics(metrics)

## 9. Clarke Error Grid Analysis

In [ ]:
# Generate Clarke Error Grid
clarke_error_grid_analysis(
    y_test,
    y_pred,
    show_plot=True,
    save_path='clarke_error_grid.png'
)

## 10. Prediction Visualization

In [ ]:
# Plot predicted vs actual
plt.figure(figsize=(12, 5))

# Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5, s=30, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('True Glucose (mg/dL)', fontsize=12)
plt.ylabel('Predicted Glucose (mg/dL)', fontsize=12)
plt.title('Predicted vs True Glucose Levels', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Residual plot
plt.subplot(1, 2, 2)
residuals = y_pred - y_test
plt.scatter(y_test, residuals, alpha=0.5, s=30, edgecolors='k', linewidths=0.5)
plt.axhline(0, color='r', linestyle='--', linewidth=2)
plt.xlabel('True Glucose (mg/dL)', fontsize=12)
plt.ylabel('Residual (mg/dL)', fontsize=12)
plt.title('Residual Plot', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Save Model for Deployment

In [ ]:
# Save model for inference
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_length': 100,
        'dropout': 0.3
    },
    'preprocessor_config': {
        'sampling_rate': sampling_rate,
        'segment_length': 1.0,
        'lowcut': 0.5,
        'highcut': 8.0
    },
    'test_metrics': metrics
}, 'cnn_gru_glucose_model.pth')

print("Model saved to: cnn_gru_glucose_model.pth")
